In [6]:
import sqlite3
import sys
import os
from dotenv import load_dotenv
import importlib
import requests
from openai import OpenAI

sys.path.append(os.path.abspath(".."))
load_dotenv(dotenv_path=os.path.join(os.path.abspath(".."), ".env"), override=True)

import src.chat_sync_core as chat_sync_core
importlib.reload(chat_sync_core)
from src.chat_sync_core import fetch_chat_messages
from src.chat_sync_core import salva_chat_su_db
import json
from src.auditor_core import valuta_chat_con_LLM, salva_valutazione_db

DB_PATH = os.path.join("..", "data", "platformhero_mirror.db")

def mia_connessione():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

USA_MODELLO_LOCALE = False  # Imposta su False se vuoi usare l'API Cloud di PlatformHero

if USA_MODELLO_LOCALE:
    
    MODEL_NAME = "llama3"
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama-local-dummy-key"
else:
    
    MODEL_NAME = "TpwgLXbi"  
    BASE_URL = "https://api.platformhero.ai/v1"
     # Utilizza automaticamente il token recuperato dal file .env

API_KEY = os.getenv("TOKEN")


client_openai = OpenAI(base_url=BASE_URL, api_key=API_KEY)







assistant_id = "X5lMM2ZTajgyIAzRitoipauS"
chat_id = "SdELgLIWfP0NjNmoEOC8flW6"

In [7]:
print("API_KEY:", API_KEY)

API_KEY: tkQGDrOqmnwtbMyT5pnxhcT8MHd2he51


In [8]:
result = fetch_chat_messages(assistant_id, chat_id, API_KEY)
print(result)

{'status': 'success', 'data': [{'object': 'Message', 'assistant_id': 'X5lMM2ZTajgyIAzRitoipauS', 'chat_id': 'SdELgLIWfP0NjNmoEOC8flW6', 'message_id': 'Mah1WS5kccCEauqni4NSDcBq', 'admin_id': 'X3ArfKYIUnvbfpNR', 'role': 'assistant', 'content': {'text': 'Sul sito ITS INCOM (pagina **Contatti**) non sono indicati **orari/giorni di disponibilità** della segreteria o del centralino: sono riportati solo i recapiti.\n\nPuoi contattarli qui:\n- **Tel. 0331 1010995**\n- **Mob. 338 8225563**\n- **Email: orientamento@itsincom.it**\n\nFonte: https://itsincom.it/contatti'}, 'num_tokens': 102, 'metadata': {}, 'created_timestamp': 1781250450934, 'updated_timestamp': 1781250450934}, {'object': 'Message', 'assistant_id': 'X5lMM2ZTajgyIAzRitoipauS', 'chat_id': 'SdELgLIWfP0NjNmoEOC8flW6', 'message_id': 'Mah1R6tkHbi2ucPM3RUPR6JX', 'admin_id': 'X3ArfKYIUnvbfpNR', 'role': 'user', 'content': {'text': 'quando sono disponibili\n'}, 'num_tokens': 6, 'metadata': {}, 'created_timestamp': 1781250435976, 'updated_ti

In [9]:
if 'result' in locals() and result.get("status") == "success":
    db_conn = mia_connessione()
    
    id_log_interno, è_aggiornato = salva_chat_su_db(db_conn, assistant_id, chat_id, result)
    
    db_conn.close()
    
    print("\n[INFO] Sincronizzazione completata con successo.")
    print(f"[INFO] Stato: {'Salvati nuovi dati / Aggiornato' if è_aggiornato else 'Nessun cambiamento (Chat identica)'}")
    print(f"[INFO] ID di riferimento interno: {id_log_interno}")
else:
    print("Errore: La variabile 'result' non contiene un payload valido. Riesegui la cella dell'API.")
    è_aggiornato = False

-> Rilevato aggiornamento per la chat SdELgLIWfP0NjNmoEOC8flW6: da 2 a 4 messaggi.
-> Chat SdELgLIWfP0NjNmoEOC8flW6 sincronizzata con successo.

[INFO] Sincronizzazione completata con successo.
[INFO] Stato: Salvati nuovi dati / Aggiornato
[INFO] ID di riferimento interno: 521a5a67-a97a-4fa9-9aff-65ea5c5ae7fd


In [ ]:
# CONTROLLO DI SICUREZZA: Blocchiamo l'esecuzione se i dati non sono cambiati
if 'è_aggiornato' in locals() and not è_aggiornato:
    print(" Controllo Idempotenza: Questa versione della chat è già stata analizzata.")
    print("Esecuzione saltata per evitare giudizi fotocopia nella tabella 'evaluations'.")

else:
    db_conn = mia_connessione()
    cursor = db_conn.cursor()
    
    cursor.execute("""
        SELECT id, messages_json FROM chat_logs 
        WHERE chat_id = ? 
        ORDER BY created_at DESC LIMIT 1;
    """, (chat_id,))

    record = cursor.fetchone()

    if record:
        log_id_db, messages_json_str = record
        elenco_messaggi = json.loads(messages_json_str)
        
        # Estrazione testo ed eliminazione bug .strip() sui dizionari
        testo_ticket_markdown = ""
        for msg in elenco_messaggi:
            ruolo = str(msg.get("role", "utente")).upper()
            contenuto_raw = msg.get("content", "")
            
            if isinstance(contenuto_raw, dict):
                contenuto_str = contenuto_raw.get("text") or contenuto_raw.get("value") or str(contenuto_raw)
            elif isinstance(contenuto_raw, list):
                contenuto_str = " ".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in contenuto_raw])
            else:
                contenuto_str = str(contenuto_raw)
                
            testo_ticket_markdown += f"--- {ruolo} ---\n{contenuto_str.strip()}\n\n"
            
        print(f"Log estratto dal DB ({len(testo_ticket_markdown)} caratteri). Invocazione Judge in cloud...")

        prompt_base = """
        Sei un assistente dell'ITS Incom Academy. Devi rispondere alle domande dell'utente basandoti esclusivamente sul sito https://itsincom.it/.

        Lo scopo di ITS INCOM Academy è quello di formare professionisti qualificati che possano inserirsi subito nel mondo del lavoro. Realizziamo percorsi formativi di Tecnico Superiore post-diploma con un forte orientamento tecnologico e rivolto alla digital transformation grazie a contenuti innovativi, concreti ed efficaci. Per queste ragioni i corsi ITS INCOM sono la migliore alternativa ai percorsi universitari ‘standard’.
        La qualità dei percorsi proposti nasce dalla sinergia di tutte le parti coinvolte che hanno un obiettivo comune: la formazione e il reperimento di lavoratori competenti (90% di assunzione degli allievi al termine dei corsi). ITS INCOM Academy quindi, quotidianamente, favorisce il confronto con le imprese, le istituzioni del tessuto produttivo, i docenti professionisti dei settori di insegnamento, lo staff ITS INCOM e i diversi enti territoriali che contribuiscono all’inserimento professionale.

        GUARDRAIL

        Non rispondere a domande che non riguardano l'ITS Incom Academy.
        """
        
        # Chiamata all'API di PlatformHero
        risposta_ai = valuta_chat_con_LLM(client_openai, MODEL_NAME, testo_ticket_markdown, prompt_base)
        
        risposta_pulita = risposta_ai.strip()
        if "{" in risposta_pulita and "}" in risposta_pulita:
            risposta_pulita = risposta_pulita[risposta_pulita.find("{"):risposta_pulita.rfind("}") + 1]
            
        # ------------------------------------------------------------------
        # PATCH UNIVERSALE ANTI-ALLUCINAZIONE SULLE METRICHE
        # ------------------------------------------------------------------
        try:
            valutazione_dati = json.loads(risposta_pulita)
            
            # Tutti e 10 i KPI devono essere presenti qui dentro:
            kpi_fondamentali = [
                "technical_score", 
                "completeness_score", 
                "business_score", 
                "consistency_score", 
                "prompt_compliance_score",
                "helpfulness_score",      
                "tone_score",             
                "hallucination_score",    
                "efficiency_score",       
                "source_reliability_score" 
            ]
            
            voti_validi = []
            for kpi in kpi_fondamentali:
                valore = valutazione_dati.get(kpi)
                if valore is not None and valore != "":
                    try:
                        voti_validi.append(float(valore))
                    except (ValueError, TypeError):
                        pass 
            
            if voti_validi:
                media_reale = round(sum(voti_validi) / len(voti_validi), 1)
            else:
                media_reale = 0.0
            
            valutazione_dati["overall_score"] = media_reale
            risposta_pulita = json.dumps(valutazione_dati, ensure_ascii=False)
            
        except Exception as e:
            print(f"[WARNING] Impossibile normalizzare le chiavi del JSON: {e}")
        
        print("\nVALUTAZIONE EMESSA (NORMALIZZATA E VERIFICATA)")
        print(risposta_pulita)
        
        # Scrittura finale sicura sul database con la chiave overall_score garantita al 100%
        salva_valutazione_db(db_conn, log_id_db, risposta_pulita)
        
    else:
        print(f"Nessun record trovato nella tabella chat_logs per il chat_id: {chat_id}")
        
    db_conn.close()

Log estratto dal DB (1389 caratteri). Invocazione Judge in cloud...

VALUTAZIONE EMESSA (NORMALIZZATA E VERIFICATA)
{"system_judicial_reasoning": "Grave mismatch di intent: alla domanda dell’utente su \"quando sono disponibili\" l’agente devia su indirizzo/requisiti/telefono senza rispondere sugli orari, fallendo copertura e chiusura del loop. Informazioni fornite appaiono in-scope e citano fonti, ma non risolvono la richiesta principale e introducono contenuti non richiesti. Tono ok e nessuna violazione di sicurezza, ma scarsa aderenza procedurale all’esigenza immediata dell’utente.", "technical_score": 7.5, "completeness_score": 2.5, "business_score": 8.0, "consistency_score": 6.0, "prompt_compliance_score": 9.5, "overall_score": 6.7, "feedback": "Risposta in-scope ma inefficace: non risponde alla domanda sugli orari e deraglia su informazioni non richieste.", "issues": "copertura_richieste, chiusura_loop, pertinenza_risposta, gestione_intent"}
Successo! Valutazione registrata. Voto 